This code is to take in embeddings extracted from a set of pre-trained models, perform dimension reduction, unsupervised clustering, and then test how the clustering across camera trap sites relates to species richness.

Specifically for the mmct dataset

In [4]:
# packages
import os
import numpy as np
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from umap import plot
import json
import hdbscan
import sklearn.cluster as cluster
from collections import defaultdict
from sklearn import metrics, datasets
from sklearn.metrics import pairwise_distances, adjusted_rand_score, adjusted_mutual_info_score
import colorcet as cc
from pathlib import Path
import re


/Users/peggybevan/anaconda3/envs/unsup/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/peggybevan/anaconda3/envs/unsup/lib/python3.10/site-packages/umap/plot.py:203: NumbaDeprecationWarning: The keyword argument 'nopython=False' was supplied. From Numba 0.59.0 the default is being changed to True and use of 'nopython=False' will raise a warning as the argument will have no effect. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit(nopython=False)


First we need to lead in the feature vectors from the output folder

In [5]:
os.chdir('/Users/peggybevan/Documents/Github/CV4E_unsupProj')

base_dir = Path('/Users/peggybevan/Documents/Github/CV4E_unsupProj')
output_dir = base_dir / 'output'
results_dir = base_dir / 'results/kenya'
results_dir.mkdir(parents=True, exist_ok=True)


In [19]:
# load metadata
#add information about camera trap location and species
meta = pd.read_csv('data/kenya/mmct_cropsmeta_PB.csv')
meta = meta.rename(columns={'location': 'ct_site'})
meta['time_hour'] = pd.to_datetime(meta['time'], format='%H:%M:%S').dt.hour
#meta has humans and vehicles in - remove
anthro = ['human', 'vehicle']
domes = ['shoat', 'domestic_dog', 'cattle']
meta = meta[-meta['species'].isin(anthro)]
#only training
meta = meta[meta['img_set']=='train']
meta_wild = meta[-meta['species'].isin(domes)]

# #create numpy array for important variables
# ct_site = np.array(meta.ct_site)
# species = np.array(meta.species)
# mgmt = np.array(meta.conservancy_name) 
# time_hour = np.array(meta.time_hour)

# # and for wild only
# ct_site_wild = np.array(meta_wild.ct_site)
# species_wild = np.array(meta_wild.species)
# mgmt_wild = np.array(meta_wild.conservancy_name)
# time_hour_wild = np.array(meta_wild.time_hour)



In [7]:
# Dimensions to try
dims_list = [128, 32, 8, 2]

# HDBSCAN hyperparameters
# I'm using a leaf cluster selection method as this allows for smaller clusters to be identified 
# whilst still allowing variable cluster size.
# we will try some settings for min cluster size based on % of total samples
n_samples = meta_wild.shape[0]
n_samples_wild = meta_wild.shape[0]


hdbscan_cfgs = {
    # changing the minimum cluster size (interpreted as 0.5%, 1%, 2% of the total sample size)
    "leaf_0p5pct": lambda n: dict(
        min_cluster_size=int(0.005 * n),
        min_samples=(int(0.005 * n_samples) // 2),
        cluster_selection_method='leaf'
    ),
    "leaf_1pct": lambda n: dict(
        min_cluster_size=int(0.01 * n),
        min_samples=(int(0.01 * n_samples) // 2),
        cluster_selection_method='leaf'
    ),
    "leaf_2pct": lambda n: dict(
       min_cluster_size=int(0.02 * n),
        min_samples=(int(0.02 * n_samples) // 2),
        cluster_selection_method='leaf'
    )
}

hdbscan_cfgs_wild = {
    # changing the minimum cluster size (interpreted as 0.5%, 1%, 2% of the total sample size)
    "leaf_0p5pct": lambda n: dict(
        min_cluster_size=int(0.005 * n),
        min_samples=(int(0.005 * n_samples_wild) // 2),
        cluster_selection_method='leaf'
    ),
    "leaf_1pct": lambda n: dict(
        min_cluster_size=int(0.01 * n),
        min_samples=(int(0.01 * n_samples_wild) // 2),
        cluster_selection_method='leaf'
    ),
    "leaf_2pct": lambda n: dict(
       min_cluster_size=int(0.02 * n),
        min_samples=(int(0.02 * n_samples_wild) // 2),
        cluster_selection_method='leaf'
    )
}


List all the models in the output folder

In [8]:
# ---- Discover models in output/ ----
feature_files = sorted(output_dir.glob('*fvect_norm.npy'))
feature_files = [f for f in feature_files if 'mmct' in f.name]
if not feature_files:
    raise FileNotFoundError(f'No files matching "*_fvect_norm.npy" found in {output_dir}')



def infer_model_name(path: Path) -> str:
    # e.g., "PegNet50_fvect_norm.npy" -> "PegNet50"
    #  or "PegNet50_wild_fvect_norm.npy" -> "PegNet50_wild"
    return re.sub(r'_fvect_norm\.npy$', '', path.name)


In [9]:
# ---- Containers for results ----
site_rows = []       # one row per (model, n_components, ct_site)
regression_rows = [] # one row per (model, n_components)
image_rows = [] #one row per image


In [10]:
# run loop over all models and UMAP dimensions
# takes 30 mins to run for 4 models x 4 dims
for fpath in feature_files:
    model = infer_model_name(fpath)
    print(f'Processing model: {model}')

    # Load feature matrix
    features = np.load(fpath)
    n_samples = features.shape[0]
    
    # if model contains "wild", use the wild metadata and HDBSCAN configs
    if 'wild' in model:
        ct_site = np.array(meta_wild.ct_site)
        species = np.array(meta_wild.species)
        mgmt = np.array(meta_wild.conservancy)
        time_hour= np.array(meta_wild.time_hour)
        datetime = np.array(meta_wild.datetime)
        hdbscan_cfgs = hdbscan_cfgs_wild
    else:
        #create numpy array for important variables
        ct_site = np.array(meta.ct_site)
        species = np.array(meta.species)
        mgmt = np.array(meta.conservancy) 
        time_hour = np.array(meta.time_hour)
        datetime = np.array(meta.datetime)

    # Sanity check: feature rows must align with meta length
    if len(ct_site) != n_samples or len(species) != n_samples:
        raise ValueError(
            f'Length mismatch for model "{model}": '
            f'features={n_samples}, ct_site={len(ct_site)}, species={len(species)}.\n'
            'Ensure your meta filtering matches the feature set order, or provide an index mapping.'
        )

    for n_comp in dims_list:
        # Build and fit UMAP
        reducer = umap.UMAP(init='random', random_state=42, n_jobs = 1, n_components=n_comp)
        embedding = reducer.fit_transform(features)
        print(f'Processing model: {model} with {n_comp} dimensions')

        # Cluster with HDBSCAN
        for cfg_name, cfg_fn in hdbscan_cfgs.items():
            # Build HDBSCAN kwargs for this dataset size
            kwargs = cfg_fn(n_samples)
            print(f'Clustering with HDBSCAN config: {cfg_name}')

            # Fit HDBSCAN
            clusterer = hdbscan.HDBSCAN(**kwargs).fit(embedding)
            labels = clusterer.labels_

            # Diagnostics
            n_clusters = int(labels.max() + 1) if labels.max() >= 0 else 0  # exclude noise (-1)
            noise_ratio = float((labels == -1).mean())

            # Save image-level cluster assignments
            # # select the right meta depending on wild or not
            meta_used = meta_wild if 'wild' in model else meta
            for i, (label, site, sp, dt) in enumerate(zip(labels, ct_site, species, datetime)):
                image_rows.append({
                    'model': model,
                    'n_components': n_comp,
                    'hdbscan_cfg': cfg_name,
                    'image_id': meta_used.index[i],   # or use a specific ID column e.g. meta_used['filename'].iloc[i]
                    'ct_site': site,
                    'species': sp,
                    'datetime': dt,
                    'cluster_label': int(label)
                })

            # Per-site counts (remove noise label -1 when counting clusters)
            distinct_label_counts = []
            distinct_species_counts = []

            for site in np.unique(ct_site):
                indices = np.where(ct_site == site)[0]
                site_labels = labels[indices]
                site_species = species[indices]

                site_labels = site_labels[site_labels != -1]
                distinct_label_count = int(len(np.unique(site_labels)))
                distinct_species_count = int(len(np.unique(site_species)))

                site_rows.append({
                    'model': model,
                    'n_components': n_comp,
                    'hdbscan_cfg': cfg_name,
                    'min_cluster_size': int(kwargs['min_cluster_size']),
                    'min_samples': int(kwargs['min_samples']),
                    'ct_site': site,
                    'distinct_label_count': distinct_label_count,
                    'distinct_species_count': distinct_species_count,
                    'n_samples_site': int(len(indices)),
                    'n_clusters_total': n_clusters,
                    'noise_ratio_total': noise_ratio
                })

                distinct_label_counts.append(distinct_label_count)
                distinct_species_counts.append(distinct_species_count)

            # Regression summaries across sites
            x = np.array(distinct_label_counts, dtype=float)
            y = np.array(distinct_species_counts, dtype=float)

            # Guard against zero-variance arrays (otherwise polyfit/corr can be nan)
            if np.all(x == x[0]) or np.all(y == y[0]):
                slope, intercept = (np.nan, np.nan)
                corr = np.nan
                r2 = np.nan
            else:
                slope, intercept = np.polyfit(x, y, 1)
                corr = np.corrcoef(x, y)[0, 1]
                # Simple r^2 from correlation
                r2 = corr**2

            regression_rows.append({
                'model': model,
                'n_components': n_comp,
                'hdbscan_cfg': cfg_name,
                'min_cluster_size': int(kwargs['min_cluster_size']),
                'min_samples': int(kwargs['min_samples']),
                'slope': float(slope) if slope == slope else np.nan,   # keep as float or NaN
                'intercept': float(intercept) if intercept == intercept else np.nan,
                'correlation_r': float(corr) if corr == corr else np.nan,
                'r_squared': float(r2) if r2 == r2 else np.nan,
                'n_sites': int(len(np.unique(ct_site))),
                'n_samples_total': int(n_samples),
                'n_clusters_global': n_clusters,
                'noise_ratio_global': noise_ratio
                })


Processing model: PegNet50_mmct
Processing model: PegNet50_mmct with 128 dimensions
Clustering with HDBSCAN config: leaf_0p5pct
Clustering with HDBSCAN config: leaf_1pct
Clustering with HDBSCAN config: leaf_2pct
Processing model: PegNet50_mmct with 32 dimensions
Clustering with HDBSCAN config: leaf_0p5pct
Clustering with HDBSCAN config: leaf_1pct
Clustering with HDBSCAN config: leaf_2pct
Processing model: PegNet50_mmct with 8 dimensions
Clustering with HDBSCAN config: leaf_0p5pct
Clustering with HDBSCAN config: leaf_1pct
Clustering with HDBSCAN config: leaf_2pct
Processing model: PegNet50_mmct with 2 dimensions
Clustering with HDBSCAN config: leaf_0p5pct
Clustering with HDBSCAN config: leaf_1pct
Clustering with HDBSCAN config: leaf_2pct
Processing model: PegNet50_mmct_wild
Processing model: PegNet50_mmct_wild with 128 dimensions
Clustering with HDBSCAN config: leaf_0p5pct
Clustering with HDBSCAN config: leaf_1pct
Clustering with HDBSCAN config: leaf_2pct
Processing model: PegNet50_mmct

In [11]:
#  ---- Build DataFrames ----
site_counts_df = pd.DataFrame(site_rows)
regression_df = pd.DataFrame(regression_rows)
image_labels_df = pd.DataFrame(image_rows)


In [12]:
regression_df

,model,n_components,hdbscan_cfg,min_cluster_size,min_samples,slope,intercept,correlation_r,r_squared,n_sites,n_samples_total,n_clusters_global,noise_ratio_global
0,PegNet50_mmct,128,leaf_0p5pct,262,131,1.569898,2.207141,0.742736,0.551657,172,52589,11,0.770979
1,PegNet50_mmct,128,leaf_1pct,525,262,2.914289,0.127560,0.571807,0.326964,172,52589,5,0.605317
2,PegNet50_mmct,128,leaf_2pct,1051,525,2.905320,0.031332,0.555695,0.308797,172,52589,5,0.603377
3,PegNet50_mmct,32,leaf_0p5pct,262,131,1.635216,1.425511,0.775039,0.600685,172,52589,11,0.740725
4,PegNet50_mmct,32,leaf_1pct,525,262,2.965028,-0.316606,0.568659,0.323373,172,52589,5,0.580920
...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,regnet128_mmct_wild,8,leaf_1pct,410,205,1.757741,-0.307004,0.586033,0.343434,172,41046,8,0.613458
92,regnet128_mmct_wild,8,leaf_2pct,820,410,1.912393,0.680283,0.610148,0.372281,172,41046,7,0.707596
93,regnet128_mmct_wild,2,leaf_0p5pct,205,102,0.737275,3.253746,0.638195,0.407293,172,41046,17,0.620304
94,regnet128_mmct_wild,2,leaf_1pct,410,205,1.365531,0.226829,0.586777,0.344307,172,41046,10,0.490718


In [13]:

site_csv = results_dir / 'clustercountsbysite.csv'
regr_csv = results_dir / 'regression_summary.csv'
image_csv = results_dir / 'image_cluster_labels.csv'

site_counts_df.to_csv(site_csv, index=False)
regression_df.to_csv(regr_csv, index=False)
image_labels_df.to_csv(image_csv, index=False) 


Choose the 'best' performing model and save the cluster/image labels

In [2]:
results_dir

NameError: name 'results_dir' is not defined

In [6]:
regr_csv = results_dir / 'regression_summary.csv'
regression_df = pd.read_csv(regr_csv)

In [14]:
noise_weight = 0.8   # increase to penalise noise more heavily
 
regression_df['balance_score'] = (
    regression_df['correlation_r'] -
    noise_weight * regression_df['noise_ratio_global']
)
 
best_row = regression_df.loc[regression_df['balance_score'].idxmax()]

In [27]:
regression_df.sort_values(by = 'balance_score', ascending=False)

,model,n_components,hdbscan_cfg,min_cluster_size,min_samples,slope,intercept,correlation_r,r_squared,n_sites,n_samples_total,n_clusters_global,noise_ratio_global,balance_score
40,convnextL_mmct_wild,32,leaf_1pct,410,205,1.242819,1.513939,0.765677,0.586261,172,41046,13,0.346879,0.488174
34,convnextL_mmct,2,leaf_1pct,525,205,1.025037,1.420519,0.740245,0.547962,172,52589,17,0.336781,0.470820
43,convnextL_mmct_wild,8,leaf_1pct,410,205,1.163534,1.944648,0.788992,0.622508,172,41046,14,0.436486,0.439803
46,convnextL_mmct_wild,2,leaf_1pct,410,205,0.981072,2.040541,0.745061,0.555116,172,41046,16,0.399527,0.425439
70,efficientNet2_mmct_wild,2,leaf_1pct,410,205,1.840355,-1.472824,0.645276,0.416381,172,41046,9,0.317424,0.391336
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,regnet128_mmct,128,leaf_1pct,525,205,1.971102,-0.904213,0.631826,0.399204,172,52589,8,0.735344,0.043551
54,efficientNet2_mmct,8,leaf_0p5pct,262,102,1.576050,1.936683,0.615081,0.378325,172,52589,9,0.727966,0.032708
80,regnet128_mmct,8,leaf_2pct,1051,410,2.258453,-1.098410,0.607042,0.368500,172,52589,7,0.753028,0.004620
77,regnet128_mmct,32,leaf_2pct,1051,410,3.125928,-1.281062,0.575566,0.331276,172,52589,5,0.739984,-0.016421


In [16]:
best_row

model                 convnextL_mmct_wild
n_components                           32
hdbscan_cfg                     leaf_1pct
min_cluster_size                      410
min_samples                           205
slope                            1.242819
intercept                        1.513939
correlation_r                    0.765677
r_squared                        0.586261
n_sites                               172
n_samples_total                     41046
n_clusters_global                      13
noise_ratio_global               0.346879
balance_score                    0.488174
Name: 40, dtype: object

In [17]:
output_dir

PosixPath('/Users/peggybevan/Documents/Github/CV4E_unsupProj/output')

In [28]:
# ---- Extract best parameters ----
best_model      = best_row['model']
best_n_comp     = 8
best_cfg        = best_row['hdbscan_cfg']
best_min_cs     = int(best_row['min_cluster_size'])
best_min_samp   = int(best_row['min_samples'])


# ---- Select correct metadata ----
if 'wild' in best_model:
    meta_used = meta_wild
else:
    meta_used = meta
 
ct_site   = np.array(meta_used.ct_site)
species   = np.array(meta_used.species)
timestamps = np.array(meta_used.datetime)   # adjust column name if needed
 
# ---- Load features ----
fpath = output_dir / f'{best_model}_fvect_norm.npy'
if not fpath.exists():
    raise FileNotFoundError(f'Feature file not found: {fpath}')
 
features = np.load(fpath)
n_samples = features.shape[0]
 
print(f'\nLoaded features: {features.shape}')


Loaded features: (41046, 1000)


In [29]:
# Sanity check
if len(ct_site) != n_samples:
    raise ValueError(
        f'Length mismatch: features={n_samples}, metadata={len(ct_site)}. '
        'Check that metadata filtering matches the feature set.'
    )

In [30]:
meta_used.columns

Index(['Unnamed: 0', 'X', 'image_name', 'box_count', 'x', 'x1', 'y1', 'x2',
       'y2', 'species', 'tagger', 'stage', 'filepath', 'label', 'ml_filtering',
       'width', 'height', 'area', 'img_id', 'box_name', 'box_path', 'img_name',
       'img_path', 'date', 'datetime', 'time', 'img_height', 'img_width',
       'ct_site', 'conservancy', 'need_to_move_directory', 'img_set',
       'boxes_per_img_id', 'category_id', 'is_in_train_1perc',
       'is_in_train_5perc', 'is_in_train_10perc', 'is_in_train_25perc',
       'is_in_train_50perc', 'is_in_train_75perc', 'box_path_PB', 'time_hour'],
      dtype='object')

In [31]:
# ---- Rerun UMAP ----
print(f'Running UMAP (n_components={best_n_comp})...')
reducer = umap.UMAP(init='random', random_state=42, n_jobs=1, 
                    n_components=best_n_comp)
embedding = reducer.fit_transform(features)
 
# ---- Rerun HDBSCAN ----
print(f'Running HDBSCAN (min_cluster_size={best_min_cs}, min_samples={best_min_samp})...')
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=best_min_cs,
    min_samples=best_min_samp,
    cluster_selection_method='leaf'
).fit(embedding)
 
labels = clusterer.labels_
 
n_clusters = int(labels.max() + 1) if labels.max() >= 0 else 0
noise_ratio = float((labels == -1).mean())
print(f'Clusters found: {n_clusters}  |  Noise ratio: {noise_ratio:.3f}')
 
# ---- Build image-level dataframe ----
# Use filename or index as image_id - adjust 'filename' to your actual column
id_col = 'image_name' if 'image_name' in meta_used.columns else None
 
image_rows = []
for i, (label, site, sp, dt) in enumerate(zip(labels, ct_site, species, timestamps)):
    row = {
        'model': best_model,
        'n_components': best_n_comp,
        'hdbscan_cfg': best_cfg,
        'image_id': meta_used[id_col].iloc[i] if id_col else meta_used.index[i],
        'ct_site': site,
        'species': sp,
        'datetime': dt,
        'cluster_label': int(label)
    }
    image_rows.append(row)
 
image_labels_df = pd.DataFrame(image_rows)
 
# ---- Save ----
out_path = results_dir / f'image_cluster_labels_{best_model}_umap{best_n_comp}_{best_cfg}.csv'
image_labels_df.to_csv(out_path, index=False)
 
print(f'\nSaved {len(image_labels_df)} rows to:')
print(f'  {out_path}')
print('\nColumn summary:')
print(image_labels_df.dtypes)
print(f'\nCluster label distribution (top 10):')
print(image_labels_df['cluster_label'].value_counts().head(10))
 

Running UMAP (n_components=8)...
Running HDBSCAN (min_cluster_size=410, min_samples=205)...
Clusters found: 14  |  Noise ratio: 0.436

Saved 41046 rows to:
  /Users/peggybevan/Documents/Github/CV4E_unsupProj/results/kenya/image_cluster_labels_convnextL_mmct_wild_umap8_leaf_1pct.csv

Column summary:
model            object
n_components      int64
hdbscan_cfg      object
image_id         object
ct_site          object
species          object
datetime         object
cluster_label     int64
dtype: object

Cluster label distribution (top 10):
cluster_label
-1     17916
 13     3981
 5      3411
 6      3235
 9      2312
 10     2258
 3      1430
 12     1329
 11     1227
 4       831
Name: count, dtype: int64


--- SCRIPT END ----

In [ ]:
# plot - ignore for now

sns.set(style="whitegrid", context="talk")

# Ensure categorical hue/style mapping (so seaborn uses discrete colors/dashes)
for df in (site_counts_df, regression_df):
    df["n_components"] = df["n_components"].astype("category")
    df["hdbscan_cfg"] = df["hdbscan_cfg"].astype("category")


In [36]:

limits_df = (
    site_counts_df.groupby(["model", "n_components", "hdbscan_cfg"])
    .agg(xmin=("distinct_label_count", "min"), xmax=("distinct_label_count", "max"))
    .reset_index()
)

lines_base = regression_df.merge(
    limits_df, on=["model", "n_components", "hdbscan_cfg"], how="left"
)

# Expand each line to 50 points between xmin and xmax
line_rows = []
for _, row in lines_base.iterrows():
    if any(pd.isna(row[k]) for k in ["slope", "intercept", "xmin", "xmax"]):
        continue
    # Skip degenerate ranges
    if row["xmax"] <= row["xmin"]:
        continue
    xs = np.linspace(row["xmin"], row["xmax"], 50)
    ys = row["slope"] * xs + row["intercept"]
    line_rows.append(pd.DataFrame({
        "model": row["model"],
        "n_components": row["n_components"],
        "hdbscan_cfg": row["hdbscan_cfg"],
        "x": xs,
        "y": ys
    }))
line_points_df = pd.concat(line_rows, ignore_index=True) if line_rows else pd.DataFrame()

# Optional: consistent dash patterns for style levels
style_order = ["leaf_0p5pct", "leaf_1pct", "leaf_2pct"]
dashes_map = {
    "leaf_0p5pct": (2, 2),   # short dash
    "leaf_1pct": (6, 2),     # medium dash
    "leaf_2pct": (10, 3),    # long dash
}


/var/folders/d3/rn8l2fts0xxbcbrzl7m0hvv00000gn/T/ipykernel_64342/2199339047.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  site_counts_df.groupby(["model", "n_components", "hdbscan_cfg"])


In [37]:

# ----- Scatter facets -----
g = sns.relplot(
    data=site_counts_df,
    x="distinct_label_count", y="distinct_species_count",
    col="model", col_wrap=3,
    hue="n_components", palette="tab10",
    style="hdbscan_cfg", style_order=style_order,
    kind="scatter", height=4, aspect=1.15,
    facet_kws={"sharex": False, "sharey": False},
    alpha=0.4
)

# After relplot
# Remove scatter points from the relplot facetsn lines on each facet -----
for ax in g.axes.flatten(): g.col_names):
    for coll in list(ax.collections):odel"] == model]
        coll.remove()

# ----- Overlay regression lines on each facet -----            data=sub,
for i, (ax, model) in enumerate(zip(g.axes.flatten(), g.col_names)):            x="x", y="y",
    sub = line_points_df[line_points_df["model"] == model]
    if not sub.empty:rder,
        sns.lineplot(
            data=sub,=2, ax=ax,
            x="x", y="y",lse  # keep legend from the scatter only
            hue="n_components", palette="tab10",
            style="hdbscan_cfg", style_order=style_order,}")
            dashes=dashes_map,")
            linewidth=2, ax=ax,
            legend=(i == 0)  # build legend once
        )ut using public seaborn API
    ax.set_title(f"{model}")
    ax.set_xlabel("Distinct Clusters per CT site")ove_legend(g, "lower right", title="n_components / HDBSCAN")
    ax.set_ylabel("Species Count per CT site")eError):

# Move legend to figure-level (no private _legend access)
handles, labels = g.axes.flatten()[0].get_legend_handles_labels()plt.show()
if handles:
    dedup = {}
    for h, l in zip(handles, labels):        if l and l not in dedup:
            dedup[l] = h
    g.figure.legend(
        dedup.values(), dedup.keys(),
        title="n_components / HDBSCAN",
        loc="lower right"
    )
    if g.axes.flatten()[0].legend_ is not None:
        g.axes.flatten()[0].legend_.remove()

SyntaxError: unmatched ')' (1180502740.py, line 15)